# This code does everything needed for volunteer monthly hour log

In [1]:

### Start of the app

import gspread
import pandas as pd
from gspread_dataframe import set_with_dataframe
import numpy as np
import matplotlib.pyplot as plt

# 1. Authenticate using JSON key
gc = gspread.service_account(filename="../python-project.json")

# 2. Open the Google Sheet
sh2 = gc.open_by_url(
    "")
# 3. Select the specific worksheet (if there are many)
worksheet_volunteer_success_applications = sh2.worksheet("volunteer-hour-report")
worksheet_members_hours = sh2.worksheet("volunteer-hour-report")

ModuleNotFoundError: No module named 'gspread'

In [ ]:
df_volunteers = pd.DataFrame(worksheet_volunteer_success_applications.get_all_records())
df_hours= pd.DataFrame(worksheet_members_hours.get_all_records())
df_volunteers.head()

,Name,Your Last Name,Creation Log,Email,Volunteering Date,Total Volunteering Hours Completed Today,Please describe the volunteering activities completed today,log type,training type,role training type,activity type
0,Romie Jr,Arias,"Kevin Jun 22, 2025 1:33 PM",maryannarias1121@gmail.com,2025-06-22,1,Workplace Safety Training Module-- OHSA,Training,Mandatory training,,
1,Romie Jr,Arias,"Kevin Jun 23, 2025 9:21 PM",maryannarias1121@gmail.com,2025-06-23,2,Training for AODA and Anti-bullying,Training,Mandatory training,,
2,Romie Jr,Arias,"Kevin Jun 28, 2025 8:28 PM",maryannarias1121@gmail.com,2025-06-28,0.5,Learning how to use monday.com for work manage...,Training,Mandatory training,,
3,Romie Jr,Arias,"Kevin Jul 1, 2025 1:46 PM",maryannarias1121@gmail.com,2025-07-01,0.5,Training for Google apps,Training,Mandatory training,,
4,Romie Jr,Arias,"Kevin Jul 6, 2025 1:01 PM",maryannarias1121@gmail.com,2025-07-06,0.5,Session 1,Training,Mandatory training,,


In [ ]:
df_hours.head()

,Name,Your Last Name,Creation Log,Email,Volunteering Date,Total Volunteering Hours Completed Today,Please describe the volunteering activities completed today,log type,training type,role training type,activity type
0,Romie Jr,Arias,"Kevin Jun 22, 2025 1:33 PM",maryannarias1121@gmail.com,2025-06-22,1,Workplace Safety Training Module-- OHSA,Training,Mandatory training,,
1,Romie Jr,Arias,"Kevin Jun 23, 2025 9:21 PM",maryannarias1121@gmail.com,2025-06-23,2,Training for AODA and Anti-bullying,Training,Mandatory training,,
2,Romie Jr,Arias,"Kevin Jun 28, 2025 8:28 PM",maryannarias1121@gmail.com,2025-06-28,0.5,Learning how to use monday.com for work manage...,Training,Mandatory training,,
3,Romie Jr,Arias,"Kevin Jul 1, 2025 1:46 PM",maryannarias1121@gmail.com,2025-07-01,0.5,Training for Google apps,Training,Mandatory training,,
4,Romie Jr,Arias,"Kevin Jul 6, 2025 1:01 PM",maryannarias1121@gmail.com,2025-07-06,0.5,Session 1,Training,Mandatory training,,


## Data cleaning

In [ ]:
df_volunteers["Email"] = df_volunteers["Email"].str.strip().str.lower()
df_hours["Email"] = df_hours["Email"].str.strip().str.lower()

In [ ]:
# Convert column Creation log to datetime for future visualizations
df_volunteers["datetime_str"] = df_volunteers["Creation Log"].str.extract(r'([A-Z][a-z]{2}\s+\d{1,2},\s+\d{4}\s+\d{1,2}:\d{2}\s+[AP]M)') #like Sep 26, 2025 3:07 AM
df_volunteers["date_joined_vps"] = pd.to_datetime(df_volunteers["datetime_str"])


df_hours["Volunteering Date"] = pd.to_datetime(df_hours["Volunteering Date"], errors="coerce")

/tmp/ipykernel_307102/1847251559.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_volunteers["date_joined_vps"] = pd.to_datetime(df_volunteers["datetime_str"])


0       6.0
1       6.0
2       6.0
3       7.0
4       7.0
       ... 
3690    3.0
3691    3.0
3692    4.0
3693    6.0
3694    NaN
Name: Volunteering Date, Length: 3695, dtype: float64

## Task 1. Who participated in volunteering and for how many hours

In [ ]:
df_volunteers["all_emails"] = df_volunteers.apply(
    lambda row: {e for e in [row["Email"], row["Email"]] if pd.notna(e)},
    axis=1
)
email_to_vol = {}

for idx, emails in df_volunteers["all_emails"].items():
    for email in emails:
        email_to_vol[email] = idx

df_hours["volunteer_index"] = df_hours["Email"].map(email_to_vol)
df_hours_matched = df_hours.dropna(subset=["volunteer_index"]).copy()
df_hours_matched["volunteer_index"] = df_hours_matched["volunteer_index"].astype(int)
merged = df_hours_matched.merge(
    df_volunteers,
    left_on="volunteer_index",
    right_index=True,
    how="left"
).reset_index(drop=True)

merged

,Name_x,Your Last Name_x,Creation Log_x,Email_x,Volunteering Date_x,Total Volunteering Hours Completed Today_x,Please describe the volunteering activities completed today_x,log type_x,training type_x,role training type_x,...,Volunteering Date_y,Total Volunteering Hours Completed Today_y,Please describe the volunteering activities completed today_y,log type_y,training type_y,role training type_y,activity type_y,datetime_str,date_joined_vps,all_emails
0,Romie Jr,Arias,"Kevin Jun 22, 2025 1:33 PM",maryannarias1121@gmail.com,2025-06-22,1,Workplace Safety Training Module-- OHSA,Training,Mandatory training,,...,2025-07-05,3,I made a video on how to cook a salmon veggie ...,Service,,,Virtual: social media,"Jul 5, 2025 5:46 PM",2025-07-05 17:46:00,{maryannarias1121@gmail.com}
1,Romie Jr,Arias,"Kevin Jun 23, 2025 9:21 PM",maryannarias1121@gmail.com,2025-06-23,2,Training for AODA and Anti-bullying,Training,Mandatory training,,...,2025-07-05,3,I made a video on how to cook a salmon veggie ...,Service,,,Virtual: social media,"Jul 5, 2025 5:46 PM",2025-07-05 17:46:00,{maryannarias1121@gmail.com}
2,Romie Jr,Arias,"Kevin Jun 28, 2025 8:28 PM",maryannarias1121@gmail.com,2025-06-28,0.5,Learning how to use monday.com for work manage...,Training,Mandatory training,,...,2025-07-05,3,I made a video on how to cook a salmon veggie ...,Service,,,Virtual: social media,"Jul 5, 2025 5:46 PM",2025-07-05 17:46:00,{maryannarias1121@gmail.com}
3,Romie Jr,Arias,"Kevin Jul 1, 2025 1:46 PM",maryannarias1121@gmail.com,2025-07-01,0.5,Training for Google apps,Training,Mandatory training,,...,2025-07-05,3,I made a video on how to cook a salmon veggie ...,Service,,,Virtual: social media,"Jul 5, 2025 5:46 PM",2025-07-05 17:46:00,{maryannarias1121@gmail.com}
4,Romie Jr,Arias,"Kevin Jul 6, 2025 1:01 PM",maryannarias1121@gmail.com,2025-07-06,0.5,Session 1,Training,Mandatory training,,...,2025-07-05,3,I made a video on how to cook a salmon veggie ...,Service,,,Virtual: social media,"Jul 5, 2025 5:46 PM",2025-07-05 17:46:00,{maryannarias1121@gmail.com}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3690,Mario,Gimigliano,"Kevin Mar 14, 2025 5:17 PM",mario.gimigliano@yorkeducation.ca,2025-03-14,3,I created 4 YRES and 3 U+ Word Searches + Solu...,Service,,,...,2025-03-14,3,I created 4 YRES and 3 U+ Word Searches + Solu...,Service,,,Virtual: web design,"Mar 14, 2025 5:17 PM",2025-03-14 17:17:00,{mario.gimigliano@yorkeducation.ca}
3691,Selena,He,"Kevin Mar 11, 2025 11:31 AM",349101998@gapps.yrdsb.ca,2025-03-11,2,I created 5 YRES coloring sheets.,Service,,,...,2025-03-11,2,I created 5 YRES coloring sheets.,Service,,,Virtual: teaching assistant,"Mar 11, 2025 11:31 AM",2025-03-11 11:31:00,{349101998@gapps.yrdsb.ca}
3692,Mane,Muradyan,"Kevin Apr 6, 2025 8:27 PM",mane.dyan@gmail.com,2025-04-06,2.5,Created 6 YRES Word searches + Solutions,Service,,,...,2025-04-06,2.5,Created 6 YRES Word searches + Solutions,Service,,,Virtual: web design,"Apr 6, 2025 8:27 PM",2025-04-06 20:27:00,{mane.dyan@gmail.com}
3693,Araneya,Mohanakumar,"Kevin Jun 29, 2025 6:30 PM",araneya.mohanakumar@yorkeducation.ca,2025-06-29,1.5,Training resources,Training,Mandatory training,,...,2025-06-29,1.5,Training resources,Training,Mandatory training,,,"Jun 29, 2025 6:30 PM",2025-06-29 18:30:00,{araneya.mohanakumar@yorkeducation.ca}


In [ ]:
merged.columns.tolist()


['Name_x',
 'Your Last Name_x',
 'Creation Log_x',
 'Email_x',
 'Volunteering Date_x',
 'Total Volunteering Hours Completed Today_x',
 'Please describe the volunteering activities completed today_x',
 'log type_x',
 'training type_x',
 'role training type_x',
 'activity type_x',
 'volunteer_index',
 'Name_y',
 'Your Last Name_y',
 'Creation Log_y',
 'Email_y',
 'Volunteering Date_y',
 'Total Volunteering Hours Completed Today_y',
 'Please describe the volunteering activities completed today_y',
 'log type_y',
 'training type_y',
 'role training type_y',
 'activity type_y',
 'datetime_str',
 'date_joined_vps',
 'all_emails']

In [ ]:
merged["Volunteering Date_x"] = pd.to_datetime(
    merged["Volunteering Date_x"], errors="coerce"
)
merged["Total Volunteering Hours Completed Today_x"] = pd.to_numeric(
    merged["Total Volunteering Hours Completed Today_x"], errors="coerce"
)


dec_mask = merged["Volunteering Date_x"].dt.month == 12
report = merged.groupby("volunteer_index").agg(
    Name=("Name_x", "first"),
    Email=("Email_x", "first"),
    date_joined_vps=("date_joined_vps", "first"),

    # November only
    training_hours_nov=("Total Volunteering Hours Completed Today_x",
                        lambda x: x[(merged.loc[x.index, "log type_x"] == "Training") &
                                    dec_mask.loc[x.index]].sum()),

    service_hours_nov=("Total Volunteering Hours Completed Today_x",
                       lambda x: x[(merged.loc[x.index, "log type_x"] == "Service") &
                                   dec_mask.loc[x.index]].sum()),

    # All time
    total_training_hours=("Total Volunteering Hours Completed Today_x",
                          lambda x: x[merged.loc[x.index, "log type_x"] == "Training"].sum()),

    total_service_hours=("Total Volunteering Hours Completed Today_x",
                         lambda x: x[merged.loc[x.index, "log type_x"] == "Service"].sum()),
).reset_index(drop=True)

report

,Name,Email,date_joined_vps,training_hours_nov,service_hours_nov,total_training_hours,total_service_hours
0,Nicole,nicole.agyekum@yorkeducation.ca,2025-08-06 12:25:00,0.0,0.0,13.0,0.00
1,Ruby,ruby.yeaman-park@yorkeducation.ca,2025-03-02 11:57:00,0.0,0.0,0.0,1.00
2,Mina,mina.lee@yorkeducation.ca,2025-03-04 11:47:00,0.0,0.0,0.0,3.00
3,Mustafa,mustafa.ismaeel@yorkeducation.ca,2025-03-06 17:13:00,0.0,0.0,0.0,2.00
4,Sachin,sachinnavaneethan@gmail.com,2025-03-09 17:20:00,0.0,0.0,0.0,1.00
...,...,...,...,...,...,...,...
652,Mario,mario.gimigliano@yorkeducation.ca,2025-03-14 17:17:00,0.0,0.0,0.0,35.00
653,"He, Selena",349101998@gapps.yrdsb.ca,2025-03-11 11:31:00,0.0,0.0,0.0,12.00
654,Mane,mane.dyan@gmail.com,2025-04-06 20:27:00,0.0,0.0,0.0,18.00
655,Araneya,araneya.mohanakumar@yorkeducation.ca,2025-06-29 18:30:00,0.0,0.0,7.0,6.25


In [ ]:
report.to_excel('report_volunteering_hours_per_person_dec.xlsx', index=False)

## Task 2. Total volunteering hours.

In [ ]:
report["total_hours"] = (
    report["total_training_hours"].fillna(0) +
    report["total_service_hours"].fillna(0)
)
report["total_hours"]

0      13.00
1       1.00
2       3.00
3       2.00
4       1.00
       ...  
652    35.00
653    12.00
654    18.00
655    13.25
656     0.00
Name: total_hours, Length: 657, dtype: float64

In [ ]:
bins = [0, 40, 80, 120, float("inf")]
labels = ["0–40 hrs", "40–80 hrs", "80–120 hrs", "120+ hrs"]

report["hour_range"] = pd.cut(report["total_hours"], bins=bins, labels=labels, right=False)
report["hour_range"].head()

range_counts = report["hour_range"].value_counts().sort_index()
range_counts


hour_range
0–40 hrs      489
40–80 hrs      92
80–120 hrs     33
120+ hrs       43
Name: count, dtype: int64

In [ ]:
summary = range_counts.rename("Volunteers").reset_index()
summary.columns = ["Hour range", "Volunteers"]

summary

,Hour range,Volunteers
0,0–40 hrs,489
1,40–80 hrs,92
2,80–120 hrs,33
3,120+ hrs,43


In [ ]:
summary.to_excel('tables/total_hours_table.xlsx', index=False)

In [ ]:
report["total_hours"].describe()


count    516.000000
mean      34.889735
std       44.269936
min      -12.000000
25%        6.000000
50%       16.225000
75%       43.500000
max      261.500000
Name: total_hours, dtype: float64